In [ ]:
!pip install pypdf
!pip install python-docx

# Import **Libraries**

In [ ]:
import json
import re
import sys
#import subprocess
import torch
import transformers
#from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path
from pypdf import PdfReader
from docx import Document
from openai import OpenAI
from google.colab import userdata
import pandas as pd
from IPython.display import HTML, display

# ***Function To Scan Through all the resume present in directory***

In [ ]:
def _get_resumes(directory_path):
    """
    Scans the given directory for resume files based on common extensions.
    """
    target_dir = Path(directory_path)

    # Check if directory exists
    if not target_dir.exists() or not target_dir.is_dir():
        print(f"Error: The directory '{directory_path}' does not exist.")
        return []

    # Define common resume file extensions
    valid_extensions = {'.pdf', '.docx'}

    resume_files = []

    # Iterate through files in the directory
    for file_path in target_dir.iterdir():
        if file_path.is_file() and file_path.suffix.lower() in valid_extensions:
            resume_files.append(file_path)
            print(file_path)

    return resume_files

## ***Extract Raw Text for pdf resumes***

In [ ]:
def _get_pdf_rawText(resume_path):
  reader = PdfReader(resume_path)
  number_of_pages = len(reader.pages)
  raw_text  = ""
  combined_payload = ""
  print('Total No of Pages: ',number_of_pages)
  for pg in range(number_of_pages):
    #print(pg)
    page = reader.pages[pg]
    raw_text = raw_text+page.extract_text()
  combined_payload += f"\n<resume path='{resume_path}'>\n{raw_text}\n</resume>\n"
  #print(raw_text)
  return combined_payload


## ***Extract Raw Text for Docx Resumes***

In [ ]:
def _get_doc_rawText(resume_path):
  doc = Document(resume_path)
  raw_text = ""
  combined_payload = ""
  for para in doc.paragraphs:
    raw_text = raw_text + para.text
  #print(raw_text)
  combined_payload += f"\n<resume path='{resume_path}'>\n{raw_text}\n</resume>\n"
  return combined_payload

# ***Clean Text to minimize tokenization***

***Remove White Space***

In [ ]:
def _remove_white_space(resume_raw_text):
  # Remove excessive whitespace
  cleaned_text = re.sub(r'[ \t]+', ' ', resume_raw_text)
  # Remove excessive newline characters
  cleaned_text = re.sub(r'\n\s*\n', '\n', cleaned_text)
  lines = cleaned_text.split('\n')
  cleaned_lines = []
  #print(len(resume_raw_text)-len(cleaned_text))
  return cleaned_text

# ***Feed Resume Cleaned Text into Qwen LLM to get JSON***

In [ ]:

def _get_json_from_llm(cleaned_text,category):

    model_name = "Qwen/Qwen2.5-3B-Instruct"
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if category == 'Resume':

      schema_template = """{

        "name": "string",
        "email": "string",
        "phone": "string",
        "skills": ["string"],
        "education": [{"degree":"string",
                      "Marks": "string"}],
        "experience": [{"company":"string",
                      "designation": "string",
                      "start_date": "string",
                      "end_date": "string"
                      }]

      }"""

      prompt = f"""Parse cleaned_text and generate json object stricly follow schema: {schema_template}\n\n
                  use resume as only context strictly no other source or history\n\n
                  generate only json object no explanation or other text\n\n

      resume: {cleaned_text}
      """
    else:

      schema_template = """{
        "Summary": "string",
        "Skills": "[string]",
        "Experience": "[string]"
      }"""

      prompt = f"""Parse cleaned_text and generate json object strictly follow schema: {schema_template}\n\n
                  use jd as only context strictly no other source or history\n\n
                  generate only valid json object no explanation or other text\n\n


      jd: {cleaned_text}
      """
    tokenized_prompt = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    input_len = tokenized_prompt[0].shape
    #print(input_len[0])
    outputs = model.generate(tokenized_prompt, do_sample=False,top_p=None, return_dict_in_generate=False, output_scores=False,max_new_tokens=1000)
    generated_tokens = outputs[:, input_len[0]:]
    #print(outputs)
    if category == 'Resume':
      resume = tokenizer.decode(generated_tokens, skip_special_tokens=True)[0]
       #print(resume)
      return resume
    else:
      jd = tokenizer.decode(generated_tokens, skip_special_tokens=True)[0]
      #print(jd)
      return jd

# ***Feed Resume Json object and Job Description into OpenAI model to get the Ranking***

In [ ]:
def rank_resumes_with_openai(resume_json_array, job_criteria):
  # Initialize the OpenAI client
  api_key = userdata.get('OPENAI')
  client = OpenAI(api_key=api_key)
  print(client)
  resumes_payload = json.dumps(resume_json_array, indent=2)

  rank_schema = """[{"name": "string",
                  "rank": "string",
                  "score": "string",
                  "justification": "string"
                  }]"""
  rule = f"""(
              1.justification must be in 50 charachters maximum
              2.Rank starts from 1, highest score must have minimum rank

              )"""

  system_prompt = f"""(
          You are an expert technical recruiter\n\n
          Scan through JSON array containing multiple candidate resumes\n\n
          Your task is to evaluate each candidate and rank them out of scale of 10 for provided job description\n\n
          follow the rules strictly: {rule}\n\n
          return valid json object only strictly following schema: {rank_schema}\n\n
          No text or explanation needed

      )"""

  user_prompt = (
      f"Job Evaluation Criteria:\n{job_criteria}\n\n"
      f"Candidate Resumes Data:\n{resumes_payload}\n\n"
      "Please analyze, score, and rank these candidates strictly as a JSON object."
  )

  try:
      # 5. Call gpt-4o-mini using JSON mode
      response = client.chat.completions.create(
          model="gpt-4o-mini",
          messages=[
              {"role": "system", "content": system_prompt},
              {"role": "user", "content": user_prompt}
          ],
          response_format={"type": "json_object"}
      )

      # 6. Parse and return the resulting dictionary
      ranking_content = response.choices[0].message.content
      return json.loads(ranking_content)

  except Exception as e:
      print(f"Error during OpenAI ranking call: {e}")
      return None

# ***Generate Report Output File From JSON***

In [ ]:
def generate_ranking_report(
    ranking_json_data,
    job_details_data=None,
    output_filename="candidate_ranking_report.html",
):
  """Takes the JSON ranking object and optional job details, sorts candidates

  by rank lowest to highest, and outputs a professional HTML report.
  """
  # 1. Handle input format for rankings
  if isinstance(ranking_json_data, str):
    data = json.loads(ranking_json_data)
  else:
    data = ranking_json_data

  candidates_list = (
      data.get("rankings", data.get("candidates", data))
      if isinstance(data, dict)
      else data
  )

  # 2. Handle job details input format
  #job_info = {}
  if job_details_data:
    if isinstance(job_details_data, str):
      job_info = json.loads(job_details_data)
    else:
      job_info = job_details_data
  print(job_info)
  # 3. Load candidates into Pandas DataFrame and sort
  df = pd.DataFrame(candidates_list)
  if "rank" in df.columns:
    df["rank"] = pd.to_numeric(df["rank"])
    df = df.sort_values(by="rank", ascending=True)

  # 4. Design professional CSS styling and HTML layout
  html_content = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>Candidate Ranking Report</title>
        <style>
            body {{
                font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
                background-color: #f8f9fa;
                color: #333333;
                margin: 0;
                padding: 40px;
            }}
            .container {{
                max-width: 1000px;
                margin: auto;
                background: #ffffff;
                padding: 30px;
                border-radius: 12px;
                box-shadow: 0 4px 15px rgba(0, 0, 0, 0.05);
            }}
            h2 {{
                color: #1a73e8;
                margin-top: 0;
                font-size: 24px;
                border-bottom: 2px solid #f1f3f4;
                padding-bottom: 15px;
            }}
            .job-card {{
                background-color: #f1f3f4;
                border-left: 4px solid #1a73e8;
                padding: 15px 20px;
                border-radius: 6px;
                margin-bottom: 25px;
            }}
            .job-card h3 {{
                margin: 0 0 10px 0;
                color: #202124;
                font-size: 18px;
            }}
            .job-card p {{
                margin: 5px 0;
                font-size: 14px;
                color: #444444;
            }}
            .meta-info {{
                font-size: 14px;
                color: #6c757d;
                margin-bottom: 25px;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 10px;
                text-align: left;
            }}
            th {{
                background-color: #f1f3f4;
                color: #202124;
                font-weight: 600;
                padding: 12px 16px;
                border-bottom: 2px solid #dee2e6;
            }}
            td {{
                padding: 14px 16px;
                border-bottom: 1px solid #e9ecef;
                font-size: 14px;
                vertical-align: top;
            }}
            tr:hover {{
                background-color: #f8f9fa;
            }}
            .rank-badge {{
                display: inline-block;
                background-color: #e8f0fe;
                color: #1a73e8;
                font-weight: bold;
                padding: 5px 10px;
                border-radius: 20px;
                text-align: center;
            }}
            .score-pill {{
                font-weight: bold;
                color: #137333;
                background-color: #ceead6;
                padding: 4px 8px;
                border-radius: 6px;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <h2>AI Recruiter: Candidate Evaluation & Ranking Report</h2>
    """

  # Conditionally render the Job Details card if provided
  if job_info:
    job_title = job_info.get("Summary", "N/A")
    experience = job_info.get("Experience", [])
    skills = job_info.get("Skills", [])

    if isinstance(experience, list):
      exp_html = "".join([f"<li>{exp}</li>" for exp in experience])
    else:
      exp_html = f"<li>{experience}</li>"

    if isinstance(skills, list):
      skills_html = "".join([f"<li>{skill}</li>" for skill in skills])
    else:
      skills_html = f"<li>{skills}</li>"

    html_content += f"""
            <div class="job-card">
                <div class="job-header">📋 <strong>Job Summary & Requirements</strong></div>
                <div class="job-section">
                    <p class="summary-text">{job_title}</p>
                </div>
                <div class="job-grid">
                    <div class="job-column">
                        <strong>Required Skills:</strong>
                        <ul class="job-list">{skills_html}</ul>
                    </div>
                    <div class="job-column">
                        <strong>Experience & Qualifications:</strong>
                        <ul class="job-list">{exp_html}</ul>
                    </div>
                </div>
            </div>

    """

  html_content += f"""
            <div class="meta-info">
                <strong>Sorting Order:</strong> Rank (Lowest to Highest) | <strong>Total Candidates Evaluated:</strong> {len(df)}
            </div>

            <table>
                <thead>
                    <tr>
                        <th style="width: 10%;">Rank</th>
                        <th style="width: 25%;">Candidate Name</th>
                        <th style="width: 15%;">Match Score</th>
                        <th style="width: 50%;">Justification</th>
                    </tr>
                </thead>
                <tbody>
    """

  # Populate table rows dynamically from DataFrame sorting
  #for _, row in df.iterrows():
  candidates_list = next((value for value in data.values() if isinstance(value, list)), [])
  for candidate in candidates_list:
        #rank = candidate['rank']
        #name = candidate['name']
        #score = candidate['score']
        #justification = candidate['justification']
        rank = candidate.get("rank")
        name = candidate.get("name")
        score = candidate.get("score")
        justification = candidate.get("justification")

        html_content += f"""
                          <tr>
                              <td><span class="rank-badge">#{rank}</span></td>
                              <td><strong>{name}</strong></td>
                              <td><span class="score-pill">{score} / 10</span></td>
                              <td>{justification}</td>
                          </tr>
    """

  html_content += """
                </tbody>
            </table>
        </div>
    </body>
    </html>
    """

  # 5. Save the output file
  with open(output_filename, "w", encoding="utf-8") as f:
    f.write(html_content)

  print(f"Report successfully generated and saved to: {output_filename}")

  # Render directly inside Google Colab output cell
  display(HTML(html_content))

# ***Main Call***

In [ ]:
if __name__ == "__main__":
    # Replace with the actual path to your folder containing resumes
    directory_to_scan = "/content/sample_data"
    jd = "/content/jd"
    resume_raw_text = ""
    clear_text =""
    found_resumes = _get_resumes(directory_to_scan)
    resume_json_pdf = ""
    resume_json_doc = ""
    final_json_array = []
    print(f"Found {len(found_resumes)} file(s):")
    for resume in found_resumes:
        resume_raw_text = ""
        clear_text = ""
        resume_json_pdf = ""
        resume_json_doc = ""
        print(f"- {resume.name} (Full path: {resume.absolute()})")
        #print("File Extension: ",resume.suffix)
        if resume.suffix == '.pdf':
          resume_raw_text = _get_pdf_rawText(resume)
          clear_text = _remove_white_space(resume_raw_text)
          #print('Extracting pdf raw text..')
          resume_json_pdf= _get_json_from_llm(clear_text,'Resume')
          final_json_array.append(json.loads(resume_json_pdf.replace("`","").replace("json","").strip()))
          #print(final_json)
        elif resume.suffix.lower() == '.docx' and resume.name[:2].lower() != ('jd'):
          resume_raw_text = _get_doc_rawText(resume)
          clear_text = _remove_white_space(resume_raw_text)
          resume_json_doc= _get_json_from_llm(clear_text,'Resume')
          final_json_array.append(json.loads(resume_json_doc.replace("`","").replace("json","").strip()))
          #print('Extracting doc or docx raw text..')
        elif resume.suffix.lower() == '.docx' and resume.name[:2].lower() == ('jd'):
          jd_raw_text = _get_doc_rawText(resume)
          clear_text = _remove_white_space(jd_raw_text)
          jd_json= _get_json_from_llm(clear_text,'JD')
          jd_final_json = json.loads(jd_json.replace("`","").replace("json","").strip())
    #Parse JD docx file to get JSON
    #found_jd = _get_resumes(jd)

    ranking_results = rank_resumes_with_openai(final_json_array, jd_final_json)
    if ranking_results:
        #print("\n--- Final Candidate Rankings ---")
        #print(json.dumps(ranking_results, indent=4))
        generate_ranking_report(json.dumps(ranking_results, indent=4),jd_json)

/content/sample_data/Roshan_Resume.pdf
/content/sample_data/Nitish_Resume.docx
/content/sample_data/Surajit_Resume.docx
/content/sample_data/resume2.docx
/content/sample_data/Avisek_Resume.docx
/content/sample_data/resume.pdf
/content/sample_data/JD-Digital Transformation Lead.docx
/content/sample_data/Preetam_resume.docx
/content/sample_data/Abhay_Resume.pdf
/content/sample_data/Mukesh_Resume.docx
/content/sample_data/Rahul_Resume.docx
Found 11 file(s):
- Roshan_Resume.pdf (Full path: /content/sample_data/Roshan_Resume.pdf)
Total No of Pages:  2


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

- Nitish_Resume.docx (Full path: /content/sample_data/Nitish_Resume.docx)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

- Surajit_Resume.docx (Full path: /content/sample_data/Surajit_Resume.docx)


...

- Mukesh_Resume.docx (Full path: /content/sample_data/Mukesh_Resume.docx)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

- Rahul_Resume.docx (Full path: /content/sample_data/Rahul_Resume.docx)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

JSONDecodeError: Extra data: line 1 column 1064 (char 1063)

In [ ]:
jd_json
#jd_final_json = json.loads(jd_json.replace("`","").replace("json","").strip())
#jd_final_json

' {"Summary": "The job description outlines the responsibilities of a Digital Transformation & Technology Lead in the cement manufacturing industry. Key responsibilities include designing custom ERP-like solutions for various business functions such as sales, logistics, finance, engineering, mining, HR, and compliance. The role also involves integrating advanced technologies like AI, IIoT, and GenAI to enhance process efficiency and drive innovation. Additionally, the candidate is expected to ensure compliance with relevant regulations and fortify IT/OT security.", "Skills": ["SAP Expertise", "Industrial Technologies", "AI/Cloud", "Enterprise GenAI Deployment", "ML-Driven Dashboards", "RPA", "Autonomous Machinery", "Mining Optimization Systems", "Finance & Commercial Automation", "HR Automation", "NLP-Driven Reporting"], "Experience": ["Leadership & Influence", "Building Teams to Deliver ERP-Like Solutions", "Cross-Module Integration", "Pilot Emerging Tools", "Coding Frameworks (Python

In [ ]:
 ranking_results = rank_resumes_with_openai(final_json_array, jd_json)
 if ranking_results:
  generate_ranking_report(json.dumps(ranking_results, indent=4),jd_json)

{'Summary': "The role of Assistant General Manager - Fleet Maintenance (HOD) involves leading the company's fleet maintenance function to ensure maximum vehicle availability, optimal maintenance cost, statutory compliance, operational safety, and workshop excellence for a fleet of 400+ Heavy Commercial Vehicles (HMVs). The role requires a BE/Diploma in Mechanical Engineering with 15-25 years of experience in fleet management, transportation logistics, supply chain, mining, cement logistics, or heavy commercial vehicle operations. Key responsibilities include ensuring maximum fleet uptime through preventive, predictive, and breakdown maintenance, preparing and managing annual maintenance budgets, optimizing maintenance, spare parts, tires, and repair costs, leading workshop operations for quality, productivity, and efficiency, ensuring statutory compliance and fleet safety, managing vendor and stakeholder relationships, leading accident and insurance management, team leadership, and fle

Rank,Candidate Name,Match Score,Justification
#1,Nitish Kumar Sinha,9 / 10,"Strong fleet management, logistics experience."
#2,RANJEET TIWARI,8 / 10,Quality and management roles in cement sector.
#3,GMR Warora Energy Ltd,7 / 10,Logistics and supply chain management experience.
#4,Abhay Nigam,6 / 10,IT and operational experience in cement industry.
#5,Roshan Kumar Shaw,4 / 10,Some supply chain skills but lacks fleet focus.
#6,Mukesh Kumar Mund,3 / 10,Limited fleet and maintenance specific experience.
#7,Avisekh Kumar,3 / 10,Lacks direct fleet and mechanical engineering.
#8,Surajit Chakraborty,1 / 10,Incomplete resume with no relevant skills.
#9,Preetam Mukhopadhyay,1 / 10,Unspecified experience and limited skills.
#10,Rahul Ghosh,1 / 10,Sales background lacks fleet experience.


In [ ]:
jd = "/content/jd"
#Parse JD to get JSON
found_jd = _get_resumes(jd)
jd_raw_text = _get_doc_rawText(found_jd[0])
clear_text = _remove_white_space(jd_raw_text)
jd_json= _get_json_from_llm(clear_text,'JD')

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
clear_text

"\n<resume path='/content/jd/Digital Transformation Lead- JD.docx'>\nJob Description: Digital Transformation & Technology Lead\nIndustry:\xa0Cement ManufacturingRole Overview:We seek a Digital Transformation & Technology Lead\xa0to lead the end-to-end digitalization of our cement manufacturing operations. This strategic role demands expertise in SAP ERP integration, industrial automation, and cutting-edge technologies (AI, IIoT, GenAI) to drive innovation across sales, logistics, finance, engineering, mining, HR, and compliance. The ideal candidate will possess a proven ability to architect custom ERP-like solutions, nurture cross-functional teams, and deliver transformative outcomes in resource-intensive industries.Key Responsibilities:1. Strategic Digital Roadmap & Process Innovation:Sales & Customer Engagement:\xa0Redesign sales workflows via custom ERP-like solutions integrated with SAP SD modules, enabling AI-driven sales forecasting and real-time CRM analytics.Logistics & Transpo

In [ ]:
found_resumes = _get_resumes("/content/sample_data")
for resume in found_resumes:
  if resume.suffix.lower() == '.docx' and resume.name[:2].lower() == ('jd'):
    print("Parsing JD..",{resume.name.lower()})
  elif resume.suffix.lower() == '.docx' and resume.name[:2].lower() != ('jd'):
    print('Resume',resume.name)

/content/sample_data/JD-Digital Transformation Lead.docx
/content/sample_data/Abhay_Resume.pdf
/content/sample_data/Avisek_Resume.docx
Parsing JD.. {'jd-digital transformation lead.docx'}
Resume Avisek_Resume.docx


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("CUDA version:", torch.version.cuda)
else:
    print("Running on CPU")

PyTorch version: 2.11.0+cpu
CUDA available: False
Running on CPU
